In [ ]:
# Loading & cleaning pipeline.
# osmnx is required for the OpenStreetMap road-network attributes used in the
# spatial join step (Lyon and Paris). Install it via the requirements file:
#     pip install -r requirements.txt
import warnings
warnings.filterwarnings('ignore')  # silence noisy pandas/geopandas FutureWarnings

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
import numpy as np
import osmnx as ox
import matplotlib.pyplot as plt


# Summary

- [Characteristics](#Characteristics)
- [Vehicles](#Vehicles)
- [Places](#Places)
- [Users](#Users)
- [Select only Paris and Lyon data](#Select-only-Paris-and-Lyon-data)
- [Select only crashes involving MMVs and grouping by accident types](#Select-only-crashes-involving-MMVs-and-grouping-by-accident-types)
- [Spatial join with GIS Data](#Spatial-join-with-GIS-data)
- [Data cleaning](#Data-cleaning)
- [Data wrangling](#Data-wrangling)



# Characteristics

In [ ]:
# Characteristics table (one row per accident) for years 2019-2023.
# Each yearly file is downloaded directly from data.gouv.fr (BAAC database).
# low_memory=False prevents the DtypeWarning caused by mixed types in some columns.
carac2023 = pd.read_csv('https://www.data.gouv.fr/api/1/datasets/r/104dbb32-704f-4e99-a71e-43563cb604f2', sep=';', low_memory=False)
carac2022 = pd.read_csv('https://www.data.gouv.fr/api/1/datasets/r/5fc299c0-4598-4c29-b74c-6a67b0cc27e7', sep=';', low_memory=False)
carac2020 = pd.read_csv('https://www.data.gouv.fr/api/1/datasets/r/07a88205-83c1-4123-a993-cba5331e8ae0', sep=';', low_memory=False)
carac2021 = pd.read_csv('https://www.data.gouv.fr/api/1/datasets/r/85cfdc0c-23e4-4674-9bcd-79a970d7269b', sep=';', low_memory=False)
carac2019 = pd.read_csv('https://www.data.gouv.fr/api/1/datasets/r/e22ba475-45a3-46ac-a0f7-9ca9ed1e283a', sep=';', low_memory=False)

# In 2022 the accident identifier column was renamed; align it with the other years.
carac2022.rename(columns={'Accident_Id': 'Num_Acc'}, inplace=True)

# Concatenate all years into a single characteristics dataframe.
carac = pd.concat([carac2022, carac2020, carac2021, carac2019, carac2023], axis=0)

# Normalise the accident identifier to a numeric type (some rows arrive as strings).
carac['Num_Acc'] = carac['Num_Acc'].astype(str).replace(" ", "").str.replace(',', '.').astype(float)


In [ ]:
# Latitude/longitude come as French-formatted strings (comma decimal separator).
# Convert them to floats and build a GeoDataFrame in WGS84.
carac['lat'] = carac['lat'].str.replace(',', '.').astype(float)
carac['long'] = carac['long'].str.replace(',', '.').astype(float)

geometry = [Point(xy) for xy in zip(carac['long'], carac['lat'])]
carac = gpd.GeoDataFrame(carac, geometry=geometry, crs='EPSG:4326')


# Vehicles

In [ ]:
# Vehicles table (one row per vehicle involved in an accident) for years 2019-2023.
# low_memory=False suppresses the mixed-type DtypeWarning.
vehi2022 = pd.read_csv('https://www.data.gouv.fr/api/1/datasets/r/c9742921-4427-41e5-81bc-f13af8bc31a0', sep=';', low_memory=False)
vehi2021 = pd.read_csv('https://www.data.gouv.fr/api/1/datasets/r/0bb5953a-25d8-46f8-8c25-b5c2f5ba905e', sep=';', low_memory=False)
vehi2020 = pd.read_csv('https://www.data.gouv.fr/api/1/datasets/r/a66be22f-c346-49af-b196-71df24702250', sep=';', low_memory=False)
vehi2019 = pd.read_csv('https://www.data.gouv.fr/api/1/datasets/r/780cd335-5048-4bd6-a841-105b44eb2667', sep=';', low_memory=False)
vehi2023 = pd.read_csv('https://www.data.gouv.fr/api/1/datasets/r/146a42f5-19f0-4b3e-a887-5cd8fbef057b', sep=';', low_memory=False)

# Concatenate all years and align the identifier dtype with the characteristics table.
vehi = pd.concat([vehi2022, vehi2020, vehi2021, vehi2019, vehi2023], axis=0)
vehi['Num_Acc'] = vehi['Num_Acc'].astype(float)


# Places

In [ ]:
# Places table (road geometry / infrastructure information for each accident).
# low_memory=False suppresses the mixed-type DtypeWarning.
lieux2022 = pd.read_csv('https://www.data.gouv.fr/api/1/datasets/r/a6ef711a-1f03-44cb-921a-0ce8ec975995', sep=';', low_memory=False)
lieux2020 = pd.read_csv('https://www.data.gouv.fr/api/1/datasets/r/e85c41f7-d4ea-4faf-877f-ab69a620ce21', sep=';', low_memory=False)
lieux2021 = pd.read_csv('https://www.data.gouv.fr/api/1/datasets/r/8a4935aa-38cd-43af-bf10-0209d6d17434', sep=';', low_memory=False)
lieux2019 = pd.read_csv('https://www.data.gouv.fr/api/1/datasets/r/2ad65965-36a1-4452-9c08-61a6c874e3e6', sep=';', low_memory=False)
lieux2023 = pd.read_csv('https://www.data.gouv.fr/api/1/datasets/r/8bef19bf-a5e4-46b3-b5f9-a145da4686bc', sep=';', low_memory=False)

# When an accident occurs at an intersection multiple street rows can exist;
# we keep a single representative row per accident.
lieux2023 = lieux2023.drop_duplicates('Num_Acc')

lieux = pd.concat([lieux2022, lieux2020, lieux2021, lieux2019, lieux2023], axis=0)


# Users

In [ ]:
# Users table (one row per person involved in an accident: drivers, passengers, pedestrians).
# low_memory=False suppresses the mixed-type DtypeWarning on these large CSV files.
usagers2022 = pd.read_csv('https://www.data.gouv.fr/api/1/datasets/r/62c20524-d442-46f5-bfd8-982c59763ec8', sep=';', low_memory=False)
usagers2020 = pd.read_csv('https://www.data.gouv.fr/api/1/datasets/r/78c45763-d170-4d51-a881-e3147802d7ee', sep=';', low_memory=False)
usagers2021 = pd.read_csv('https://www.data.gouv.fr/api/1/datasets/r/ba5a1956-7e82-41b7-a602-89d7dd484d7a', sep=';', low_memory=False)
usagers2019 = pd.read_csv('https://www.data.gouv.fr/api/1/datasets/r/36b1b7b3-84b4-4901-9163-59ae8a9e3028', sep=';', low_memory=False)
usagers2023 = pd.read_csv('https://www.data.gouv.fr/api/1/datasets/r/68848e2a-28dd-4efc-9d5f-d512f7dbe66f', sep=';', low_memory=False)

usagers = pd.concat([usagers2022, usagers2020, usagers2021, usagers2019, usagers2023], axis=0)


# Merging the four datasets with their correspondent

In [7]:
carac3 = pd.merge(carac,lieux, on='Num_Acc')
carac3 = pd.merge(carac3, vehi, on='Num_Acc')
carac3 = pd.merge(carac3,usagers, on='id_vehicule')


# Select only Paris and Lyon data

In [8]:
paris=gpd.read_file('https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/georef-france-epci/exports/geojson?lang=fr&refine=epci_name%3A%22M%C3%A9tropole%20du%20Grand%20Paris%22&facet=facet(name%3D%22epci_name%22%2C%20disjunctive%3Dtrue)&timezone=Europe%2FBerlin')
lyon=gpd.read_file('https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/georef-france-epci/exports/geojson?lang=fr&refine=epci_name%3A%22M%C3%A9tropole%20de%20Lyon%22&facet=facet(name%3D%22epci_name%22%2C%20disjunctive%3Dtrue)&timezone=Europe%2FBerlin')

data_lyon=gpd.sjoin(carac3,lyon[['epci_name_upper','geometry']]).rename(columns={'epci_name_upper': 'epci'})
data_paris=gpd.sjoin(carac3,paris[['epci_name_upper','geometry']]).rename(columns={'epci_name_upper': 'epci'})

data_lyon = data_lyon.drop(columns=['num_veh_y','num_veh_x','Num_Acc_y','index_right'])
data_paris = data_paris.drop(columns=['num_veh_y','num_veh_x','Num_Acc_y','index_right'])

whole_dataset=pd.concat([data_paris,data_lyon])

Skipping field reg_code: unsupported OGR type: 5
Skipping field reg_name: unsupported OGR type: 5
Skipping field dep_code: unsupported OGR type: 5
Skipping field dep_name: unsupported OGR type: 5
Skipping field epci_code: unsupported OGR type: 5
Skipping field epci_current_code: unsupported OGR type: 5
Skipping field epci_name: unsupported OGR type: 5
Skipping field reg_code: unsupported OGR type: 5
Skipping field reg_name: unsupported OGR type: 5
Skipping field dep_code: unsupported OGR type: 5
Skipping field dep_name: unsupported OGR type: 5
Skipping field epci_code: unsupported OGR type: 5
Skipping field epci_current_code: unsupported OGR type: 5
Skipping field epci_name: unsupported OGR type: 5


# Select only crashes involving MMVs and grouping by accident types

In [9]:
# Select rows corresponding to drivers (catu==1) or pedestrians (catu==3)
crashes_and_vehicles = whole_dataset.loc[
    (whole_dataset['catu'] == 1) | (whole_dataset['catu'] == 3)
].copy()   # .copy() to avoid SettingWithCopyWarning when we assign new columns

# Initialize a new 'Vehicle' column with empty strings
crashes_and_vehicles['Vehicle'] = ''

# Map 'catv' vehicle categories to human-readable labels
# Use .loc[...] assignment to be explicit and avoid warnings
crashes_and_vehicles.loc[
    crashes_and_vehicles['catv'].isin([7, 10]), 'Vehicle'
] = 'Car'

crashes_and_vehicles.loc[
    crashes_and_vehicles['catv'] == 50, 'Vehicle'
] = 'E-PMD'   # or "e-PMD"

crashes_and_vehicles.loc[
    crashes_and_vehicles['catv'] == 80, 'Vehicle'
] = 'E-bike'

crashes_and_vehicles.loc[
    crashes_and_vehicles['catv'] == 1, 'Vehicle'
] = 'Bike'

crashes_and_vehicles.loc[
    crashes_and_vehicles['catv'].isin([30, 31, 32, 33, 34, 2]), 'Vehicle'
] = 'Motorcycle'

crashes_and_vehicles.loc[
    crashes_and_vehicles['catv'].isin([13, 14]), 'Vehicle'
] = 'Truck'

crashes_and_vehicles.loc[
    crashes_and_vehicles['catv'].isin([37, 38]), 'Vehicle'
] = 'Bus'

crashes_and_vehicles.loc[
    crashes_and_vehicles['catv'] == 40, 'Vehicle'
] = 'Tram'

crashes_and_vehicles.loc[
    crashes_and_vehicles['catv'].isin([41, 42, 43]), 'Vehicle'
] = 'Three-wheeled motorized'

crashes_and_vehicles.loc[
    crashes_and_vehicles['catv'] == 60, 'Vehicle'
] = 'Mechanical PMD'

# Explicitly set pedestrians based on 'catu' (category of the person)
crashes_and_vehicles.loc[
    crashes_and_vehicles['catu'] == 3, 'Vehicle'
] = 'Pedestrian'

# Remove rows where Vehicle is still empty (no relevant vehicle type)
crashes_and_vehicles = crashes_and_vehicles[crashes_and_vehicles['Vehicle'] != ''].copy()

# Group vehicles by accident ID and build a comma-separated list of vehicles involved in each accident
vehicles_per_accident = (
    crashes_and_vehicles.groupby('Num_Acc_x')['Vehicle']
    .apply(lambda x: ', '.join(x))
    .reset_index()
)

# Keep only accidents that contain at least one micromobility mode:
# make the regex flexible to match variations like "E-PMD", "E-bike", "bike"
micromobility_pattern = r'e-?\s?pmd|e-?\s?bike|bike'

filtered_accidents = vehicles_per_accident[
    vehicles_per_accident['Vehicle'].str.contains(micromobility_pattern, case=False, na=False)
].copy()

# Rename column 'Vehicle' -> 'Accident' to reflect that this column contains the list of vehicles for the accident
filtered_accidents.rename(columns={'Vehicle': 'Accident'}, inplace=True)

# Merge crashes and filtered results on accident ID
merged_data = pd.merge(crashes_and_vehicles, filtered_accidents, on='Num_Acc_x')

def replace_vehicle(row):
    """
    Remove the main vehicle (row['Vehicle']) from the list of involved vehicles
    and return the remaining vehicles as a comma-separated string.
    """
    accident_list = row['Accident'].split(', ')
    
    # Remove the main vehicle from the list
    accident_list.remove(row['Vehicle'])
    
    # Join remaining vehicles back into a string
    return ', '.join(accident_list)

# Apply the function to each row
merged_data['Involved vehicles'] = merged_data.apply(replace_vehicle, axis=1)

# Replace empty strings with NaN
merged_data['Involved vehicles'] = merged_data['Involved vehicles'].replace('', np.nan)

# Count involved vehicles (remaining ones + the main vehicle)
merged_data['number of involved vehicles'] = merged_data['Involved vehicles'].apply(
    lambda x: len(x.split(', ')) if pd.notna(x) else 0
) + 1

# Remove accidents with more than 5 involved vehicles
merged_data = merged_data.drop(
    merged_data.loc[merged_data['number of involved vehicles'] > 5].index
)

# Split the string of involved vehicles into multiple columns
vehicles_split = merged_data['Involved vehicles'].str.split(', ', expand=True)

# Add appropriate column names
vehicles_split.columns = [f'Involved vehicle {i+1}' for i in range(vehicles_split.shape[1])]

# Concatenate with original DataFrame (drop the original column)
data_acci = pd.concat([merged_data.drop(columns=['Involved vehicles']), vehicles_split], axis=1)

data_acci_selected = data_acci[data_acci['Vehicle'] != 'Pedestrian'][
    ['id_vehicule', 'Vehicle', 'Accident',
     'number of involved vehicles',
     'Involved vehicle 1', 'Involved vehicle 2',
     'Involved vehicle 3', 'Involved vehicle 4']
]

# Merge with the main dataset
data_acci = pd.merge(whole_dataset, data_acci_selected, on='id_vehicule')

# Convert catu==3 to "Pedestrian"
data_acci.loc[data_acci['catu'] == 3, 'Vehicle'] = 'Pedestrian'



# Spatial join with GIS data

In [10]:
def join_signal_features(buffered_gdf, feature_gdf, column_name):
    acc_with_feature = gpd.sjoin(buffered_gdf.drop_duplicates('Num_Acc_x'), feature_gdf.to_crs('EPSG:3035'))
    buffered_gdf[column_name] = 'no'
    buffered_gdf.loc[buffered_gdf['Num_Acc_x'].isin(acc_with_feature['Num_Acc_x']), column_name] = 'yes'
    return buffered_gdf

## Lyon

In [11]:
accidents_lyon=data_acci.loc[data_acci['epci']== 'MÉTROPOLE DE LYON']
accidents_lyon=accidents_lyon[['Num_Acc_x','int','geometry','adr']].drop_duplicates('Num_Acc_x') #Same place for all users in the crash

lyon_cycle=gpd.read_file('https://data.grandlyon.com/geoserver/metropole-de-lyon/ows?SERVICE=WFS&VERSION=2.0.0&request=GetFeature&typename=metropole-de-lyon:pvo_patrimoine_voirie.pvoamenagementcyclable&outputFormat=application/json&SRSNAME=EPSG:4171&sortBy=gid')
lyon_network=gpd.read_file('https://data.grandlyon.com/geoserver/metropole-de-lyon/ows?SERVICE=WFS&VERSION=2.0.0&request=GetFeature&typename=metropole-de-lyon:pvo_patrimoine_voirie.pvochausseetrottoir&outputFormat=application/json&SRSNAME=EPSG:4171&sortBy=gid')

# Mergging the GIS layers of the cycle networks and road networks of Lyon 
roads = lyon_network.to_crs(3035)       
cycles = lyon_cycle.to_crs(3035)        
cycles.rename(columns={'senscirculation': 'senscirculation_cycle'}, inplace=True)
roads_buf = roads.copy()
roads_buf['geometry'] = roads_buf.buffer(5)  # m
lyon_road_network = gpd.sjoin(roads_buf, cycles, how='left', predicate='intersects')

# Convert to the desired CRS before saving
lyon_road_network.to_crs(epsg=2154).to_file('lyon_road_network.geojson', driver='GeoJSON')


# assurer CRS identiques
accidents_lyon = accidents_lyon.to_crs(lyon_road_network.crs)
lyon_road_network = lyon_road_network.reset_index(drop=True)

# construire une ligne (array) par accident : shape (n_accidents, n_lyon_road_network)
dist_matrix = np.vstack([lyon_road_network.geometry.distance(pt).values for pt in accidents_lyon.geometry])


# DataFrame lignes = accidents, colonnes = lyon_road_network
dist_df = pd.DataFrame(dist_matrix, index=accidents_lyon.index, columns=lyon_road_network.index)

# transposée si besoin (lyon_road_network x accidents)
dist_transpose = dist_df.T

nearest_indices = dist_transpose.apply(lambda dist: dist.idxmin())


# Assign attributes of nearest features to accident points
nearest_features = lyon_road_network.iloc[nearest_indices]  # Flatten indices to 1D array for indexing
nearest_features_reset  = nearest_features.reset_index(drop=True)

nearest_features = nearest_features.drop(columns=['geometry'])

accidents_lyon = pd.concat([accidents_lyon.reset_index(drop=True), nearest_features.reset_index(drop=True)], axis=1)


### OSM attributes for Lyon

In [12]:
# Définir le polygone de la Métropole de Lyon
lyon_metropole_polygon = ox.geocode_to_gdf("Métropole de Lyon, France")

# Récupérer les données pour toute la métropole
lyon_traffic_signals = ox.features_from_polygon(
    lyon_metropole_polygon.geometry.iloc[0],
    tags={"highway": "traffic_signals"}
)
lyon_stops = ox.features_from_polygon(
    lyon_metropole_polygon.geometry.iloc[0],
    tags={"highway": "stop"}
)
lyon_give_way = ox.features_from_polygon(
    lyon_metropole_polygon.geometry.iloc[0],
    tags={"highway": "give_way"}
)

# Conversion du CRS
lyon_traffic_signals.to_crs(epsg=3035, inplace=True)
lyon_stops.to_crs(epsg=3035, inplace=True)
lyon_give_way.to_crs(epsg=3035, inplace=True)

accidents_lyon=accidents_lyon.to_crs('EPSG:3035')
acc_lyon_int=accidents_lyon.loc[(accidents_lyon['int']==2) | (accidents_lyon['int']== 3) | (accidents_lyon['int']==4) | (accidents_lyon['int']==5) ].drop_duplicates('Num_Acc_x')
acc_lyon_buffered = acc_lyon_int.to_crs('EPSG:3035').buffer(25)

acc_lyon_buffered_gdf = gpd.GeoDataFrame(acc_lyon_int,geometry=acc_lyon_buffered, crs='EPSG:3035')


acc_lyon_buffered_gdf = join_signal_features(acc_lyon_buffered_gdf, lyon_stops, 'stops')
acc_lyon_buffered_gdf = join_signal_features(acc_lyon_buffered_gdf, lyon_give_way, 'give_way')
acc_lyon_buffered_gdf = join_signal_features(acc_lyon_buffered_gdf, lyon_traffic_signals, 'traffic_lights')
accidents_lyon=pd.merge(accidents_lyon,acc_lyon_buffered_gdf[['Num_Acc_x','stops','give_way','traffic_lights']].drop_duplicates('Num_Acc_x'),on='Num_Acc_x',how='left')


## Paris

In [13]:
accidents_paris=data_acci.loc[data_acci['epci']== 'MÉTROPOLE DU GRAND PARIS']
accidents_paris=accidents_paris[['Num_Acc_x','int','geometry']].drop_duplicates('Num_Acc_x')

reseau_cyclable=gpd.read_file('https://data.iledefrance.fr/api/explore/v2.1/catalog/datasets/amenagements-velo-en-ile-de-france/exports/geojson?lang=fr&timezone=Europe%2FBerlin').to_crs('EPSG:2154')

paris.to_crs('EPSG:2154',inplace=True)
reseau_cyclable_paris=gpd.sjoin(reseau_cyclable,paris[['geometry','epci_name_upper']]) # Select only the cycle network overlayed on the zone of interest 

accidents_paris.to_crs('EPSG:3035',inplace=True)
reseau_cyclable_paris.to_crs('EPSG:3035',inplace=True)

distances_paris = accidents_paris.geometry.apply(lambda point: reseau_cyclable_paris.distance(point))

# Transpose the distances DataFrame
dist_transpose_paris = np.transpose(distances_paris)

# Find the index of the nearest cycling path for each accident point
nearest_indices = dist_transpose_paris.apply(lambda dist: dist.idxmin())

# Extract the attributes of the nearest cycling paths
nearest_features = reseau_cyclable.iloc[nearest_indices]

# Reset the index of nearest_features to align with the indices of accident_sur_piste
nearest_features_reset = nearest_features.reset_index(drop=True)

nearest_features = nearest_features.drop(columns='geometry')

# Assign the nearest cycling path ID to each accident point
accidents_paris = pd.concat([accidents_paris.reset_index(drop=True), nearest_features.reset_index(drop=True)], axis=1)


/opt/anaconda3/envs/tfe/lib/python3.13/site-packages/shapely/measurement.py:72: RuntimeWarning: invalid value encountered in distance
  return lib.distance(a, b, **kwargs)


### OSM attributes for Paris

In [14]:
# Définir le polygone de la Métropole du Grand Paris
grand_paris_polygon = ox.geocode_to_gdf("Métropole du Grand Paris, France")

# Récupérer les données pour toute la métropole
paris_traffic_signals = ox.features_from_polygon(
    grand_paris_polygon.geometry.iloc[0],
    tags={"highway": "traffic_signals"}
)
paris_stops = ox.features_from_polygon(
    grand_paris_polygon.geometry.iloc[0],
    tags={"highway": "stop"}
)
paris_give_way = ox.features_from_polygon(
    grand_paris_polygon.geometry.iloc[0],
    tags={"highway": "give_way"}
)

# Conversion du CRS
paris_traffic_signals.to_crs(epsg=3035, inplace=True)
paris_stops.to_crs(epsg=3035, inplace=True)
paris_give_way.to_crs(epsg=3035, inplace=True)

acc_paris_int=accidents_paris.loc[(accidents_paris['int']==2) | (accidents_paris['int']== 3) | (accidents_paris['int']==4) | (accidents_paris['int']==5)].drop_duplicates('Num_Acc_x')

acc_buffered = acc_paris_int.to_crs('EPSG:3035').buffer(25)

acc_buffered_gdf = gpd.GeoDataFrame(acc_paris_int,geometry=acc_buffered, crs='EPSG:3035')

acc_buffered_gdf = join_signal_features(acc_buffered_gdf, paris_stops, 'stops')
acc_buffered_gdf = join_signal_features(acc_buffered_gdf, paris_give_way, 'give_way')
acc_buffered_gdf = join_signal_features(acc_buffered_gdf, paris_traffic_signals, 'traffic_lights')
accidents_paris=pd.merge(accidents_paris,acc_buffered_gdf[['Num_Acc_x','stops','give_way','traffic_lights']].drop_duplicates('Num_Acc_x'),on='Num_Acc_x',how='left')


## Concatenate both sub-datasets

In [15]:
paris=pd.merge(data_acci.loc[data_acci['epci']== 'MÉTROPOLE DU GRAND PARIS'],accidents_paris.drop(columns=['int','geometry']),on='Num_Acc_x',how='left')
lyon=pd.merge(data_acci.loc[data_acci['epci']== 'MÉTROPOLE DE LYON'],accidents_lyon.drop(columns=['int','geometry']),on='Num_Acc_x',how='left')
paris.rename(columns={'Num_Acc_x':'Num_Acc'},inplace=True)
lyon.rename(columns={'Num_Acc_x':'Num_Acc'},inplace=True)
paris=paris.rename(columns={'EPCI_name_u':'ville'})

# Combine the data from Paris and Lyon into a single DataFrame
acci = pd.concat([paris, lyon])

# Data cleaning

In [16]:
acci_clean = acci.drop(columns=[
    'dep','agg','occutc','secu3','v2','v1','pr1','pr','lartpc','larrout','adr'
    'geo_shape','panneaux','revetementpiste','osm_id','petite_ech',
    'commune2_left','insee2_left','nomvoie2','notes','precisionr','continuite',
    'matieresda','itineraire','sens_voit','codefuv','insee1_left','insee1_right',
    'gid_right','itinerair0','commune1_left','commune1_right','nomvoie1','voie',
    'insee_com','codetronco','geo_point_2d','hierarchi0','revetemen0',
    'revetemen1','limitation','hierarchie','routegrand','id','nom','observation',
    'zonecirculationapaisee','commune2_right','insee2_right','financementac',
    'typeamenagement2'
], errors='ignore')


In [17]:
acci_clean = acci_clean.reset_index(drop=True)
acci_clean.index.name='index'

In [18]:
 # Replace all instances of "Sens Unique" with "UNIQUE" in column: 'senscircul'
acci_clean['senscirculation'] = acci_clean['senscirculation_cycle'].str.replace("Sens Unique", "UNIQUE", case=False, regex=False)
    # Replace all instances of "double" with "DOUBLE" in column: 'senscircul'
   
acci_clean['senscirculation'] = acci_clean['senscirculation_cycle'].str.replace("double", "DOUBLE", case=False, regex=False)


#acci_clean = acci_clean.drop(columns=['tram_crossing'])
    # Replace all instances of "z30" with "Zone 30" in column: 'nv'
acci_clean['nv'] = acci_clean['nv'].str.replace("z30", "Zone 30", case=False, regex=False)
    # Replace all instances of "z20" with "Zone de rencontre" in column: 'nv'
acci_clean['nv'] = acci_clean['nv'].str.replace("z20", "Zone de rencontre", case=False, regex=False)
    # Replace all instances of "rue pietonne" with "Aire Piétonne" in column: 'nv'
acci_clean['nv'] = acci_clean['nv'].str.replace("rue pietonne", "Aire Piétonne", case=False, regex=False)
    
acci_clean['reglementa'] = acci_clean['reglementationzca'].combine_first(acci_clean['nv'])
    # Rename column 'longueurtr' to 'longueur_trottoir_droit'
acci_clean = acci_clean.rename(columns={'longueurtr': 'longueur_trottoir_droit'})
    # Rename column 'surfacetro' to 'surface_trottoir_droit'
acci_clean = acci_clean.rename(columns={'surfacetro': 'surface_trottoir_droit'})
    # Rename column 'largeurtr0' to 'largeur_trottoir_droit'

acci_clean = acci_clean.rename(columns={'largeurtr0': 'largeur_trottoir_droit'})

acci_clean=acci_clean.rename(columns={'reseaupl': 'Truck traffic', 'revetement': 'Pavement','epci': 'Agglomeration'})
    
    # Rename column 'longueurt0' to 'longueur_trottoir_gauche'
acci_clean = acci_clean.rename(columns={'longueurt0': 'longueur_trottoir_gauche'})
    # Rename column 'surfacetr0' to 'surface_trottoir_gauche'
acci_clean = acci_clean.rename(columns={'surfacetr0': 'surface_trottoir_gauche'})





# Data wrangling

### Infrastructure

In [19]:
acci_clean.loc[(acci_clean['int']!=1) ,'Crossroad']='No traffic lights'
acci_clean.loc[(acci_clean['traffic_lights']=='yes'),'Crossroad']='Traffic lights'
acci_clean.loc[(acci_clean['stops']=='yes'),'Crossroad']='Stop sign'
acci_clean.loc[(acci_clean['int']==6),'Crossroad']='Roundabouts'
acci_clean['Crossroad'] = acci_clean['Crossroad'].fillna('No intersection')

acci_clean['Cycle facilities']='No cycle facilities'
acci_clean.loc[acci_clean['moyenn_ech'].isin(['21','22']),'Cycle facilities']='Cycle lane'
acci_clean.loc[acci_clean['moyenn_ech'].isin(['11','12']),'Cycle facilities']='Cycle path/Greenway'
acci_clean.loc[acci_clean['moyenn_ech'].isin(['21','22']),'Cycle facilities']='Cycle lane'
acci_clean.loc[acci_clean['moyenn_ech'].isin(['23','24']),'Cycle facilities']='Bus lane'
acci_clean.loc[acci_clean['moyenn_ech'].isin(['41']),'Cycle facilities']='Pedestrianized street'

acci_clean.loc[acci_clean['typeamenagement'].str.contains('Bande Cyclable', na=False),'Cycle facilities']='Cycle lane'
acci_clean.loc[acci_clean['typeamenagement'].str.contains('Piste Cyclable', na=False),'Cycle facilities']='Cycle path/Greenway'
acci_clean.loc[acci_clean['typeamenagement'].str.contains('Voie verte', na=False),'Cycle facilities']='Cycle path/Greenway'

acci_clean.loc[acci_clean['typeamenagement'].str.contains('bus', na=False),'Cycle facilities']='Bus lane'

acci_clean['positionnement_piste']=acci_clean['positionnement']
acci_clean.loc[(acci_clean['positionnement'].str.contains('unidirectionnel')) &(acci_clean['positionnement'].str.contains('unidirectionnel')),'positionnement_piste']='Unidirectionnel'
acci_clean.loc[(acci_clean['positionnement'].str.contains('bidirectionnel')) &(acci_clean['positionnement'].str.contains('bidirectionnel')),'positionnement_piste']='Bidirectionnel'
acci_clean.loc[(acci_clean['ad'].str.contains('uni')) &(acci_clean['ag'].str.contains('uni')),'positionnement_piste']='Unidirectionnel'
acci_clean.loc[(acci_clean['ad'].str.contains('uni')) | (acci_clean['ag'].str.contains('uni')),'positionnement_piste']='Unidirectionnel'
acci_clean.loc[(acci_clean['ad'].str.contains('piste uni')) &(acci_clean['ag'].str.contains('piste uni')),'positionnement_piste']='Bidirectionnel'
acci_clean.loc[(acci_clean['ad'].str.contains('bi')) | (acci_clean['ag'].str.contains('bi')),'positionnement_piste']='Bidirectionnel'
acci_clean.loc[(acci_clean['ad'].str.contains('DSC')) | (acci_clean['ag'].str.contains('DSC')),'positionnement_piste']='Bidirectionnel'

acci_clean.loc[acci_clean['Pavement'].isin(['Béton Bitumineux','asphalt']), 'Pavement',] = 'Asphalt'
acci_clean.loc[acci_clean['Pavement'].isin(['sett', 'Pavés', 'paved', 'paving_stones', 'Dalles Pavés','cobblestone']), 'Pavement'] = 'Paved'
acci_clean.loc[acci_clean['Pavement'].isin(['concrete']), 'Pavement'] = 'Concrete'
acci_clean.loc[~acci_clean['Pavement'].isin(['Asphalt','Paved','Concrete']), 'Pavement'] = 'Other'

acci_clean['Surface condition']='Other'
acci_clean.loc[acci_clean['surf'].isin([1]), 'Surface condition',] = 'Normal'
acci_clean.loc[acci_clean['surf'].isin([2, 3, 4,6]), 'Surface condition'] = 'Wet'

acci_clean['Intersection']='Other'
acci_clean.loc[acci_clean['int'].isin([1]), 'Intersection',] = 'No intersection'
acci_clean.loc[acci_clean['int'].isin([2]), 'Intersection'] = 'Cross intersection'
acci_clean.loc[acci_clean['int'].isin([3]), 'Intersection'] = 'T intersection'
acci_clean.loc[acci_clean['int'].isin([4]), 'Intersection'] = 'Y intersection'
acci_clean.loc[acci_clean['int'].isin([6]), 'Intersection'] = 'Roundabout'
acci_clean.loc[acci_clean['int'].isin([7]), 'Intersection'] = 'Square'

acci_clean['Long profile']='Other'
acci_clean.loc[acci_clean['prof'].isin([-1]), 'Long profile',] = 'Unspecified'
acci_clean.loc[acci_clean['prof'].isin([2,3,4]), 'Long profile'] = 'Slope'
acci_clean.loc[acci_clean['prof'].isin([1]), 'Long profile'] = 'Flat'


acci_clean['Road type']='Other'
acci_clean.loc[acci_clean['highway'].isin(['residential']), 'Road type'] = 'Residential'
acci_clean.loc[acci_clean['highway'].isin(['primary']), 'Road type'] = 'Primary'
acci_clean.loc[acci_clean['highway'].isin(['secondary']), 'Road type'] = 'Secondary'
acci_clean.loc[acci_clean['highway'].isin(['tertiary']), 'Road type'] = 'Tertiary'
acci_clean.loc[acci_clean['highway'].isin(['cycleway']), 'Road type'] = 'Cycleway'
acci_clean.loc[acci_clean['highway'].isin(['path']), 'Road type'] = 'Path'
acci_clean.loc[acci_clean['highway'].isin(['footway','pedestrian','living_street']), 'Road type'] = 'Pedestrian area'

acci_clean['Road width'] = pd.cut(acci_clean['largeurchaussee'], bins=[0,4,6,8,10,12,100], labels=['<4 m','4-6 m','6-8 m','8-10 m','10-12 m', '> 12m'], right=False).astype(str)
acci_clean['Road width'] = acci_clean['Road width'].fillna('Missing')

acci_clean['Reglementation']=''
acci_clean.loc[acci_clean['reglementa'].isna(), 'Reglementation'] = 'No particularity'
acci_clean.loc[acci_clean['reglementa'].isin(['Zone 30']), 'Reglementation'] = 'Zone 30'
acci_clean.loc[acci_clean['reglementa'].isin(['Aire Piétonne']), 'Reglementation'] = 'Pedestrian area'
acci_clean.loc[acci_clean['reglementa'].isin(['Zone de rencontre']), 'Reglementation'] = 'Meeting zone (Z20)'
acci_clean.loc[acci_clean['reglementa'].isin(['limite 30']), 'Reglementation'] = 'Limit 30'


acci_clean['Max speed']='Other'
acci_clean.loc[acci_clean['vma'].isin([50]), 'Max speed'] = '50 km/h'
acci_clean.loc[acci_clean['vma'].isin([30]), 'Max speed'] = '30 km/h'
acci_clean.loc[acci_clean['vma'].isin([20]), 'Max speed'] = '20 km/h'
acci_clean.loc[acci_clean['vma'].isin([1,2,3,8,6,5,10]), 'Max speed'] = '< 10 km/h'

acci_clean['Accident location']='Other'
acci_clean.loc[acci_clean['situ'].isin([1]), 'Accident location'] = 'On road'
acci_clean.loc[acci_clean['situ'].isin([4]), 'Accident location'] = 'On sidewalk'
acci_clean.loc[acci_clean['situ'].isin([5]), 'Accident location'] = 'On cycle facility'

acci_clean.loc[acci_clean['situ'].isin([6]), 'Accident location'] = 'On other special way'




### Socio-demographics

In [20]:
acci_clean['severity'] =acci_clean['grav'].replace({2: 3, 4: 2, 3:2})

acci_clean['age']=acci_clean['an']-acci_clean['an_nais']

acci_clean['Gender']='Other'
acci_clean.loc[acci_clean['sexe'].isin([1]), 'Gender'] = 'Male'
acci_clean.loc[acci_clean['sexe'].isin([2]), 'Gender'] = 'Female'

acci_clean['User category']='Other'
acci_clean.loc[acci_clean['catu'].isin([1]), 'User category'] = 'Driver'
acci_clean.loc[acci_clean['catu'].isin([2]), 'User category'] = 'Passenger'
acci_clean.loc[acci_clean['catu'].isin([3]), 'User category'] = 'Pedestrian'

bins = [0, 20, 40, 60,100]  # Intervalles d'âge
labels = ['0-20', '21-40', '41-60','61+']  # Étiquettes pour chaque intervalle

# Créer une nouvelle colonne 'Age category' contenant les catégories d'âge
acci_clean['Age category'] = pd.cut(acci_clean['age'], bins=bins, labels=labels, right=False).astype(str)

acci_clean['Helmet']='No'
acci_clean.loc[(acci_clean['secu1']==2),'Helmet']='Yes'
acci_clean.loc[(acci_clean['secu2']==2),'Helmet']='Yes'

acci_clean['Reflective jacket']='No'
acci_clean.loc[(acci_clean['secu1']==4),'Reflective jacket']='Yes'
acci_clean.loc[(acci_clean['secu2']==4),'Reflective jacket']='Yes'

acci_clean['Trip purpose']='Other'
acci_clean.loc[acci_clean['trajet'].isin([5,3]), 'Trip purpose'] = 'Leisure/Shopping'
acci_clean.loc[acci_clean['trajet'].isin([2,1,4]), 'Trip purpose'] = 'Professional use'


### Temporal

In [21]:
def determine_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    elif month in [9, 10, 11]:
        return 'Autumn'
    else:
        return 'Unknown'  # Au cas où il y aurait une valeur invalide

# Appliquer la fonction pour créer la colonne 'season'
acci_clean['season'] = acci_clean['mois'].apply(determine_season)




acci_clean['jour'] = acci_clean['jour'].astype(int)
acci_clean['mois'] = acci_clean['mois'].astype(int)
acci_clean['an'] = acci_clean['an'].astype(int)

acci_clean.rename(columns={'jour':'day','mois':'month','an':'Year'},inplace=True)

acci_clean['date'] = pd.to_datetime(acci_clean[['day', 'month', 'Year']],format='%Y-%m-%d')
acci_clean['date_str'] = acci_clean['Year'].astype(str) + '-' + acci_clean['month'].astype(str).str.zfill(2) + '-01'

# Convertir en type date
acci_clean['date2'] = pd.to_datetime(acci_clean['date_str'], format='%Y-%m-%d')

# Supprimer la colonne temporaire si nécessaire
acci_clean.drop('date_str', axis=1, inplace=True)

# Extraire le jour de la semaine
acci_clean['day_of_week'] = acci_clean['date'].dt.day_name()

acci_clean['Weekend']='No'
acci_clean.loc[acci_clean['day_of_week'].isin(['Sunday', 'Saturday']), 'Weekend'] = 'Yes'

acci_clean['hour'] = acci_clean['hrmn'].apply(lambda x: int(str(x).zfill(4)[:2]))

def time_of_day(hour):
    if 7 <= hour < 9:
        return 'Morning peak hours'

    elif 17 <= hour < 19:
        return 'Afternoon peak hours'
    else:
        return 'Offpeak'
    

acci_clean['Time of day'] = acci_clean['hour'].apply(time_of_day)

acci_clean['Weather conditions']='Other'
acci_clean.loc[acci_clean['atm'].isin([1]), 'Weather conditions'] = 'Normal'
acci_clean.loc[acci_clean['atm'].isin([2,3,4]), 'Weather conditions'] = 'Rain / Snow'
acci_clean.loc[acci_clean['atm'].isin([8,5]), 'Weather conditions'] = 'Cloudy / Fog'

acci_clean['Lighting conditions']='Other'
acci_clean.loc[acci_clean['lum'].isin([1]), 'Lighting conditions'] = 'Daylight'
acci_clean.loc[acci_clean['lum'].isin([2]), 'Lighting conditions'] = 'Twilight'
acci_clean.loc[acci_clean['lum'].isin([3,4]), 'Lighting conditions'] = 'Night without street lightings'

acci_clean.loc[acci_clean['lum'].isin([5]), 'Lighting conditions'] = 'Night with street lightings on'

### Pedestrian 

In [22]:
acci_clean['Pedestrian action'] = 'Other'
acci_clean.loc[acci_clean['actp'].isin(['3']), 'Pedestrian action'] = 'Crossing'
acci_clean.loc[acci_clean['actp'].isin(['2','1']), 'Pedestrian action'] = 'Moving'

acci_clean['Pedestrian localisation']='Other'
acci_clean.loc[acci_clean['locp'].isin([-1]), 'Pedestrian localisation',] = 'Unknown'
acci_clean.loc[acci_clean['locp'].isin([0]), 'Pedestrian localisation',] = 'No information'
acci_clean.loc[acci_clean['locp'].isin([1,2]), 'Pedestrian localisation',] = 'On the road'
acci_clean.loc[acci_clean['locp'].isin([4, 3]), 'Pedestrian localisation'] = 'On a pedestrian crossing'
acci_clean.loc[acci_clean['locp'].isin([5]), 'Pedestrian localisation'] = 'On sidewalk'


In [23]:
acci_clean['Point of impact']='Other'
acci_clean.loc[acci_clean['choc'].isin([1,2,3]), 'Point of impact',] = 'Front'
acci_clean.loc[acci_clean['choc'].isin([4, 5,6]), 'Point of impact'] = 'Back'
acci_clean.loc[acci_clean['choc'].isin([7]), 'Point of impact'] = 'Right side'
acci_clean.loc[acci_clean['choc'].isin([8]), 'Point of impact'] = 'Left side'
acci_clean.loc[acci_clean['choc'].isin([9]), 'Point of impact'] = 'Multiple'

acci_clean.loc[acci_clean['choc'].isin([0]), 'Point of impact'] = 'No'

### Passengers

In [24]:
# Step 1: Filter for passengers
passenger_data = acci_clean[acci_clean['catu'] == 2]

# Step 2: Count the number of passengers per accident
passenger_counts = passenger_data.groupby('id_vehicule').size().reset_index(name='Number of passengers')

passenger_counts['Number of passengers'] =passenger_counts['Number of passengers'].astype(str)
# Step 3: Merge the counts with the original dataframe
acci_clean = pd.merge(acci_clean, passenger_counts, on='id_vehicule', how='left')

# Fill NaN values with 0 for accidents without passengers
acci_clean['Number of passengers'] = acci_clean['Number of passengers'].fillna(0)

passengers = acci_clean.loc[acci_clean['User category']=='Passenger']
drivers = acci_clean.loc[acci_clean['User category']=='Driver']

df_merged=pd.merge(passengers, drivers,on='id_vehicule', suffixes=('', '_driver'))

# Filter out rows where the id_usager is the same
df_filtered = df_merged[df_merged['id_usager'] != df_merged['id_usager_driver']]

# Select relevant columns
df_filtered = df_filtered[['id_usager', 'age_driver','Helmet_driver','Gender_driver','severity_driver']]

acci_clean['id_usager'].fillna(999,inplace=True)


# Merge back with the original dataframe to include age_2, vehicle_type_2, and Maneuver_2
acci_clean = pd.merge(acci_clean, df_filtered[['id_usager', 'age_driver','Helmet_driver','Gender_driver','severity_driver']], how='left',on='id_usager')

acci_clean[['id_usager', 'age_driver','Helmet_driver','Gender_driver','severity_driver']]=acci_clean[['id_usager', 'age_driver','Helmet_driver','Gender_driver','severity_driver']].fillna('999')

acci_clean['Gender_driver'] = acci_clean['Gender_driver'].astype(str)
acci_clean['Helmet_driver'] = acci_clean['Helmet_driver'].astype(str)



/var/folders/y0/0nrj3m412p978185q3d2503sr02q24/T/ipykernel_69123/911964579.py:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  acci_clean['id_usager'].fillna(999,inplace=True)


### Maneuvers

In [ ]:
# Group the raw maneuver codes (column 'manv') into a small set of human-readable
# categories. Working on a copy ensures pandas does not raise SettingWithCopyWarning
# when downstream cells continue to assign new columns.
acci_clean = acci_clean.copy()
acci_clean['Maneuver'] = 'Other'
acci_clean.loc[acci_clean['manv'].isin([5]), 'Maneuver'] = 'In the opposite direction'
acci_clean.loc[acci_clean['manv'].isin([1, 2]), 'Maneuver'] = 'Without change of direction'
acci_clean.loc[acci_clean['manv'].isin([15]), 'Maneuver'] = 'Turning left'
acci_clean.loc[acci_clean['manv'].isin([16]), 'Maneuver'] = 'Turning right'
acci_clean.loc[acci_clean['manv'].isin([24]), 'Maneuver'] = 'Parked'
acci_clean.loc[acci_clean['manv'].isin([19]), 'Maneuver'] = 'Crossing the road'
acci_clean.loc[acci_clean['manv'].isin([22]), 'Maneuver'] = 'Door opened'
acci_clean.loc[acci_clean['manv'].isin([13, 14]), 'Maneuver'] = 'Swerving'
acci_clean.loc[acci_clean['manv'].isin([17, 18]), 'Maneuver'] = 'Overtaking'


### Obstacle

In [26]:
acci_clean['Obstacle']='Other'
acci_clean.loc[acci_clean['obs'].isin([1]), 'Obstacle'] = 'Parked_vehicle'
acci_clean.loc[acci_clean['obs'].isin([8]), 'Obstacle'] = 'Poteau'
acci_clean.loc[acci_clean['manv'].isin([12]), 'Obstacle'] = 'Sidewalk bordure'




### Second-party vehicle

In [27]:

acci_clean['Vehicle type']=''
acci_clean.loc[acci_clean['Vehicle'].isin(['Truck','Tram','Bus']), 'Vehicle type'] = 'Large motorized vehicle'
acci_clean.loc[acci_clean['Vehicle'].isin(['Car']), 'Vehicle type'] = 'Cars'
acci_clean.loc[acci_clean['Vehicle'].isin(['Motorcycle','Three-wheeled motorized']), 'Vehicle type'] = 'Light motorized vehicle'
acci_clean.loc[acci_clean['Vehicle'].isin(['Pedestrian']), 'Vehicle type'] = 'Pedestrian'
acci_clean.loc[acci_clean['Vehicle'].isin(['Mechanical PMD','Bike', 'E-bike','E-PMD',]), 'Vehicle type'] = 'Micromobility vehicle'

In [28]:
# Define a danger ranking for the vehicle types
danger_ranking = {
    'Large motorized vehicle': 1,
    'Cars': 2,
    'Light motorized vehicle': 3,
    'Micromobility vehicle':4

}

# Assign the danger ranking to each row
acci_clean['danger_rank'] = acci_clean['Vehicle type'].map(danger_ranking)
acci_clean.loc[acci_clean['Vehicle'] == 'Pedestrian', 'id_vehicule'] = range(len(acci_clean.loc[acci_clean['Vehicle'] == 'Pedestrian']))


driver=acci_clean.loc[acci_clean['catu']!=2]
df_merged = pd.merge(driver, driver, on='Num_Acc', suffixes=('', '_opposite'))

# Filter out rows where the id_vehicule is the same
df_filtered = df_merged[df_merged['id_vehicule'] != df_merged['id_vehicule_opposite']]

# Select relevant columns
df_filtered = df_filtered[['Num_Acc', 'id_vehicule', 'id_vehicule_opposite','age_opposite', 'Vehicle_opposite', 'Vehicle type_opposite', 'Maneuver_opposite', 'Gender_opposite', 'severity_opposite', 'Point of impact_opposite']]

# Rename columns for clarity
df_filtered.rename(columns={
    'age_opposite': 'age_2', 
    'Vehicle_opposite': 'Vehicle_2',
    'Vehicle type_opposite': 'vehicle_type_2', 
    'Maneuver_opposite': 'Maneuver_2',
    'Gender_opposite': 'Gender_2',
    'severity_opposite': 'severity_2',
    'Point of impact_opposite': 'Point of impact_2',
    'id_vehicule_opposite': 'id_vehicule_opposite_2'
}, inplace=True)

# Assign danger ranking to the opposite vehicle type
df_filtered['danger_rank_2'] = df_filtered['vehicle_type_2'].map(danger_ranking)

# Sort by Num_Acc, id_vehicule, and danger_rank_2
df_filtered = df_filtered.sort_values(by=['Num_Acc', 'id_vehicule', 'danger_rank_2'])

# Drop duplicates to keep the row with the most dangerous vehicle
df_filtered = df_filtered.drop_duplicates(subset=['Num_Acc', 'id_vehicule'])

# Merge back with the original dataframe to include age_2, vehicle_type_2, and Maneuver_2
acci_clean = pd.merge(acci_clean, df_filtered[['Num_Acc', 'id_vehicule', 'age_2', 'vehicle_type_2', 'Vehicle_2', 'Maneuver_2', 'Gender_2', 'severity_2', 'Point of impact_2','id_vehicule_opposite_2']], on=['Num_Acc', 'id_vehicule'], how='left')

acci_clean['vehicle_type_2']=acci_clean['vehicle_type_2'].fillna('No other vehicle')
acci_clean['Maneuver_2'] = acci_clean['Maneuver_2'].fillna('Missing')


In [29]:
bins = [0, 20, 40, 60,100]  # Intervalles d'âge
labels = ['0-20', '21-40', '41-60','61+']  # Étiquettes pour chaque intervalle

# Créer une nouvelle colonne 'Age category' contenant les catégories d'âge
acci_clean['Age category involved'] = pd.cut(acci_clean['age_2'], bins=bins, labels=labels, right=False).astype(str)

### Third-party vehicle

In [30]:

# Repeat the process for the third vehicle
driver=acci_clean.loc[acci_clean['catu']!=2]

df_merged = pd.merge(driver, driver, on='Num_Acc', suffixes=('', '_opposite'))

df_filtered_3 = df_merged[(df_merged['id_vehicule'] != df_merged['id_vehicule_opposite'])]


df_filtered_3 = df_filtered_3[df_filtered_3['id_vehicule'] != df_filtered_3['id_vehicule_opposite']]
df_filtered_3 = df_filtered_3[ (df_filtered_3['id_vehicule_opposite_2'] != df_filtered_3['id_vehicule_opposite'])]

df_filtered_3 = df_filtered_3[['Num_Acc', 'id_vehicule', 'id_vehicule_opposite','age_opposite', 'Vehicle_opposite', 'Vehicle type_opposite', 'Maneuver_opposite', 'Gender_opposite', 'severity_opposite', 'Point of impact_opposite']]
df_filtered_3.rename(columns={
    'age_opposite': 'age_3', 
    'Vehicle_opposite': 'Vehicle_3',
    'Vehicle type_opposite': 'vehicle_type_3', 
    'Maneuver_opposite': 'Maneuver_3',
    'Gender_opposite': 'Gender_3',
    'severity_opposite': 'severity_3',
    'Point of impact_opposite': 'Point of impact_3',
    'id_vehicule_opposite': 'id_vehicule_opposite_3'
}, inplace=True)
df_filtered_3['danger_rank_3'] = df_filtered_3['vehicle_type_3'].map(danger_ranking)
df_filtered_3 = df_filtered_3.sort_values(by=['Num_Acc', 'id_vehicule', 'danger_rank_3'])
df_filtered_3 = df_filtered_3.drop_duplicates(subset=['Num_Acc', 'id_vehicule'])
acci_clean = pd.merge(acci_clean, df_filtered_3[['Num_Acc', 'id_vehicule','age_3', 'vehicle_type_3', 'Vehicle_3', 'Maneuver_3', 'Gender_3', 'severity_3', 'Point of impact_3','id_vehicule_opposite_3']], on=['Num_Acc', 'id_vehicule'], how='left')

In [ ]:
# Build the average opponent age. For single-vehicle crashes there is no opponent,
# so we use 0; for two-vehicle crashes we take the second driver's age, and for
# more vehicles we average the ages of the second and third drivers.
acci_clean.loc[acci_clean['number of involved vehicles'] == 1, 'age_opposite_mean'] = 0
acci_clean.loc[acci_clean['number of involved vehicles'] == 2, 'age_opposite_mean'] = acci_clean['age_2']
acci_clean.loc[acci_clean['number of involved vehicles'] > 2, 'age_opposite_mean'] = (acci_clean['age_2'] + acci_clean['age_3']) / 2

# 999 is the sentinel used by Biogeme for missing data. Avoid the chained-inplace
# FutureWarning by reassigning the column instead of using inplace=True.
acci_clean['age_3'] = acci_clean['age_3'].fillna(999)
acci_clean['age_2'] = acci_clean['age_2'].fillna(999)


# Load the final dataset

In [ ]:
# Persist the cleaned dataset for the descriptive-statistics and modeling notebooks.
# index=False removes the unnamed index column that otherwise reappears as a 'mixed
# types' column when the file is re-read by pandas.
acci_clean.to_csv('final_processed_crash_dataset_2.csv', index=False)


In [52]:
##To plot on a GIS software
acci_clean.loc[acci_clean['Vehicle'].isin(['Pedestrian', 'E-PMD', 'E-bike', 'Bike'])].to_file('map_crash.geojson')
